In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import butter, filtfilt, iirnotch

In [ ]:
class RawSignal:
    """
    Clase para manejar señales fisiológicas en formato NumPy.
    Este constructor permite inicializar el objeto 'RawSignal' a partir de un array de datos ,
    con información adicional de los canales y el índice de la primera muestra.
    """
    
    def __init__(self, data, sfreq, info=None, anotaciones=None, first_samp=0):
        """
        Inicializa una instancia de la clase RawSignal.
        
        Parameters
        ----------
        data : np.ndarray
            Matriz de datos con forma '(n_canales , n_muestras)'.
        sfreq : float
            Frecuencia de muestreo de la señal en Hz.
        info: Objeto del tipo  info. Opcional. Por defecto es None.
            Información adicional sobre la señal. El diccionario contiene info relevante de la señal
        anotaciones : Anotaciones
            Objeto de tipo Anotaciones que almacena eventos asociados a la señal y al experimento.
        first_samp : int, optional
            Índice del primer muestreo a utilizar (default es 0).
        
        Raises
        ------
        ValueError
            Si el array 'data' no tiene la forma '(n_canales , n_muestras)'.
        ValueError
            Si el índice 'first_samp' está fuera del rango de la señal.
        """
        # Validar tipo de data
        if not isinstance(data, np.ndarray): ##que data sea un array de NumPy
            raise ValueError("El parámetro 'data' debe ser un array de NumPy (np.ndarray).")

        if data.ndim != 2: ##que data tenga dos dimensiones 
            raise ValueError("El array 'data' debe tener dos dimensiones: (n_canales, n_muestras).")

        n_muestras = data.shape[1] # Número de muestras es la segunda dimensión
        if not (0 <= first_samp < n_muestras): ##que first_samp sea un entero positivo y menor que el numero de muestras
            raise ValueError("El índice 'first_samp' está fuera del rango de muestras disponibles.")

        #Asignación de atributos
        self.data = data
        self.sfreq = sfreq
        self.info = info
        self.anotaciones = anotaciones
        self.first_samp = first_samp
        
    def get_data(self, picks=None, start=0, stop=0, reject=None, times=False):
        """
        Obtiene muestras de la señal en un rango dado.

        Parameters
        ----------
        picks : str o array_like, optional
            Canales o índices a extraer. Si es 'None', se seleccionan todos los canales.
        start : float, optional
            Tiempo inicial (en segundos) para extraer muestras (por defecto 0).
        stop : float, optional
            Tiempo final (en segundos) para extraer muestras (por defecto 0, que significa hasta el final de la señal).
        reject : float, optional
            Valor pico a pico de umbral para rechazar canales. Si una muestra supera este umbral, el canal se descarta (por defecto 'None').
        times : bool, optional
            Si es 'True', se retorna también el vector de tiempos asociado a las muestras.

        Returns
        -------
        np.ndarray
            Matriz con los datos seleccionados (n_canales x n_muestras).
        np.ndarray (opcional)
            Vector de tiempos (solo si 'times=True').

        Raises
        ------
        ValueError
            Si los índices seleccionados están fuera de rango.
        """
        
        n_canales, n_muestras = self.data.shape # Número de canales y muestras  
    
        # Convertir start y stop de segundos a índices de muestra
        start_idx = int(start * self.sfreq)
        stop_idx = int(stop * self.sfreq) if stop > 0 else n_muestras #si stop es mayor que 0, convertir a 
                                                                    #indice de muestra, sino, tomar hasta el final

        if not (0 <= start_idx < stop_idx <= n_muestras): ## si no se cumple que start_idx es mayor o igual a 0 y 
                                                          ## menor que stop_idx y ademas stop_idx es menor o igual 
                                                          ## al numero de muestras, lleva al error
            raise ValueError("Índices de tiempo fuera de rango.")

        # Seleccionar canales
        if picks is None: ##Si picks queda por defecto, seleccionar todos los canales
            canales_idx = np.arange(n_canales)
        elif isinstance(picks, (list, np.ndarray)): ##Si picks es lista o un array, creo una variable canales_idx
                                                    ##que convierte la lista o array (depicks) a un array de NumPy
            canales_idx = np.array(picks)
        elif isinstance(picks, str):
            # Si info está disponible, buscar canal por nombre
            if self.info and "canales" in self.info: ##busca dentro de la lista de los canales el nombre str del canal
                                                     ##que se determino en el parametro picks y si no lo encuentra,
                                                     #lanza un error indicando que el canal no existe.
                canales_idx = [i for i, ch in enumerate(self.info["canales"]) if ch == picks]
                if not canales_idx:
                    raise ValueError(f"Canal '{picks}' no encontrado.")
            else:
                raise ValueError("No se puede buscar por nombre sin metadatos de canales.")
        else:
            raise ValueError("Formato de 'picks' inválido. Debe ser None, str o lista de índices.")

        # Extraer datos seleccionados
        datos = self.data[np.array(canales_idx), start_idx:stop_idx]

        # Aplicar umbral de rechazo si se especifica
        if reject is not None:
            p2p = np.ptp(datos, axis=1)  # pico a pico por canal
            mask = p2p < reject ##creo mascara booleana, true es debajo del umbral y false es arriba del umbral
            datos = datos[mask] ##me quedo con los canales que cumplen la condicion de la mascara

        if times:
            tiempo_vector = np.arange(start_idx, stop_idx) / self.sfreq ##
            return datos, tiempo_vector

        return datos
    
    def drop_channels(self, ch_names) -> "RawSignal":
        """
        Elimina uno o más canales a partir de ch_names
        Parameters
        ----------
        ch_names:array_like
        Nombres de canales a eliminar
        Returns
        ----------
        RawSignal
        """

        # Validaciones
        if self.info is None or "canales" not in self.info:
            raise ValueError("No se puede eliminar canales sin info['canales'].")

        if isinstance(ch_names, str):
            ch_names = [ch_names]
        elif not isinstance(ch_names, (list, np.ndarray)):
            raise ValueError("'ch_names' debe ser una lista, array o string.")

        # Lista original de canales
        canales_actuales = self.info["canales"]

        # Verificar que todos los canales existan
        for ch in ch_names:
            if ch not in canales_actuales:
                raise ValueError(f"El canal '{ch}' no existe en info['canales'].")

        # Crear máscara para mantener los canales no eliminados
        canales_a_mantener = [i for i, nombre in enumerate(canales_actuales) if nombre not in ch_names]

        # Filtrar data y actualizar info
        nueva_data = self.data[canales_a_mantener, :]
        nueva_info = self.info.copy()
        nueva_info["canales"] = [canales_actuales[i] for i in canales_a_mantener]

        # Crear y devolver nueva instancia de RawSignal
        return RawSignal(
            data=nueva_data,
            sfreq=self.sfreq,
            info=nueva_info,
            anotaciones=self.anotaciones,
            first_samp=self.first_samp
        )
        
    def crop(self, tmin=0.0, tmax=None)->"RawSignal":
        """
        Obtiene un trozo (Crop) de RawSignal. Limita los datos dentro de RawSignal
        para obtener un nuevo objeto RawSignal pero con una cantidad de muestras recortadas.
        El parámetro 'first_samp' se configura adecuadamente.
        
        Parameters
        ------------
        tmin : float , optional
            Tiempo inicial , en segundos , para iniciar el recorte (por defecto es 0.0).
        tmax : float or None , optional
            Tiempo final , en segundos , para finalizar el recorte (por defecto es None).
        
        Returns
        -----------
        RawSignal
            Nueva instancia de 'RawSignal' que contiene el segmento temporal recortado.
        
        Raises
        -----------
        Value Error
            Si los tiempos 'tmin' o 'tmax' están fuera del rango de la señal.
        """
        
        n_canales, n_muestras = self.data.shape
        duracion_total = n_muestras / self.sfreq

        #Validaciones
        if not isinstance(tmin, (int, float)) or tmin < 0: ## si tmin no es entero, flotante o positivo
            raise ValueError("'tmin' debe ser un número positivo.")
        if tmax is not None and (not isinstance(tmax, (int, float)) or tmax <= tmin): #si tmax no fue definido por 
                                                                                #el usuario Y ADEMAS, no es un 
                                                                                # numero o ese numero es mayor o 
                                                                                # igual a tmin...
            raise ValueError("'tmax' debe ser mayor que 'tmin'.")

        if tmin > duracion_total: ##Chequeo si tmin es mayor que la duracion total de la señal 
            raise ValueError(f"'tmin' está fuera del rango de la señal ({duracion_total:.2f} s).")
        if tmax is not None and tmax > duracion_total: ##si tmax fue  definido por el usuario 
                                                         #y es mayor que la duracion total de la señal
            raise ValueError(f"'tmax' está fuera del rango de la señal ({duracion_total:.2f} s).")

        # Convertir a índices
        start_idx = int(tmin * self.sfreq)
        end_idx = int(tmax * self.sfreq) if tmax is not None else n_muestras

        # Extraer el segmento de la señal
        datos_crop = self.data[:, start_idx:end_idx]

        # Ajustar first_samp
        new_first_samp = self.first_samp + start_idx

        # Crear y retornar nueva instancia
        return RawSignal(
            data=datos_crop,
            sfreq=self.sfreq,
            info=self.info,
            anotaciones=self.anotaciones,
            first_samp=new_first_samp
        )

    def describe(self):
        """
        Genera un DataFrame con estadísticas descriptivas para cada canal.

        Returns
        -------
        pd.DataFrame
            Tabla con los siguientes campos por canal:
            - name: nombre del canal
            - type: tipo de canal (ej. eeg, ecg, emg)
            - min: valor mínimo
            - Q1: primer cuartil (25%)
            - mediana: mediana (50%)
            - Q3: tercer cuartil (75%)
            - max: valor máximo
        """
    
        n_canales = self.data.shape[0]
        stats = []

        # Obtener nombres y tipo de canal desde self.info
        nombres = self.info["canales"] if self.info and "canales" in self.info else [f"Canal_{i}" for i in range(n_canales)]
        tipo = self.info["tipo"] if self.info and "tipo" in self.info else "desconocido"

        for i in range(n_canales):
            canal = self.data[i]
            fila = {
                "name": nombres[i],
                "type": tipo,
                "min": np.min(canal),
                "Q1": np.percentile(canal, 25),
                "mediana": np.median(canal),
                "Q3": np.percentile(canal, 75),
                "max": np.max(canal)
            }
            stats.append(fila)

        return pd.DataFrame(stats)
    
    def filter(self, l_freq, h_freq, notch_freq=50., order=4, fir_window="hamming") -> "RawSignal":
        """
        Aplica un filtro pasabanda y un filtro notch.
        Retorna una nueva instancia de RawSignal con los datos filtrados.

        Parameters
        ----------
        l_freq : float
            Frecuencia de corte baja (Hz) para el filtro pasabanda.
        h_freq : float
            Frecuencia de corte alta (Hz) para el filtro pasabanda.
        notch_freq : float, optional
            Frecuencia del filtro notch para eliminar ruido (por defecto 50 Hz).
        order : int, optional
            Orden del filtro (por defecto 4).
        fir_window : str, optional
            Tipo de ventana para el diseño del filtro FIR (por defecto "hamming").

        Returns
        -------
        RawSignal
            Nueva instancia de 'RawSignal' con los datos filtrados.

        Raises
        ------
        ValueError
            Si los valores de 'l_freq', 'h_freq' o 'notch_freq' no son válidos.           
        Value Error
            Si el valor de 'notch_freq' no es positivo.
        """
    
        # Validaciones
        if not (0 < l_freq < h_freq < self.sfreq / 2): ##si no se cumple esa condicion, lanza un error
            raise ValueError(f"Valores inválidos: asegúrese de que 0 < l_freq < h_freq < Nyquist ({self.sfreq / 2} Hz).")

        if notch_freq is not None and notch_freq <= 0: #si la frecuencia de notch fue definida por el usuario 
                                                       #y es menor o igual a 0
            raise ValueError("notch_freq debe ser positivo.")

        nyq = self.sfreq / 2  # Frecuencia de Nyquist para que no haya espejo de aliasing

        # Aplicar filtro pasabanda Butterworth usando scypy.signal.butter
        b_band, a_band = butter(N=order, Wn=[l_freq / nyq, h_freq / nyq], btype='band')
        data_filtrada = filtfilt(b_band, a_band, self.data, axis=1)

        # Aplicar filtro notch si se especifica
        if notch_freq is not None:
            q = 30.0  # Factor de calidad típico
            b_notch, a_notch = iirnotch(notch_freq / nyq, q)
            data_filtrada = filtfilt(b_notch, a_notch, data_filtrada, axis=1)

        # Crear nueva instancia de RawSignal con los datos filtrados
        return RawSignal(
            data=data_filtrada,
            sfreq=self.sfreq,
            info=self.info,
            anotaciones=self.anotaciones,
            first_samp=self.first_samp
        )

    def pick(self, picks) -> "RawSignal":
        """
        Retorna un subset de canales seleccionados.

            Parameters
        ----------
        picks : str | list[str] | list[int] | slice
            Canales a seleccionar. Puede ser:
            - str : nombre de un solo canal
            - list[str] : lista de nombres de canales
            - list[int] : lista de índices de canales
            - slice : rango de índices de canales

        Returns
        -------
        RawSignal
            Nueva instancia con los canales seleccionados.

        Raises
        ------
        ValueError
            Si el canal especificado no existe.
            Si el índice está fuera del rango de canales.
        """
        
        n_canales = self.data.shape[0]

        # Obtener nombres de canales si están disponibles
        nombres_canales = self.info["canales"] if self.info and "canales" in self.info else None

        # Determinar índices a seleccionar
        if isinstance(picks, str):  # Un solo nombre
            if not nombres_canales:
                raise ValueError("No se puede seleccionar por nombre sin info['canales'].")
            if picks not in nombres_canales:
                raise ValueError(f"Canal '{picks}' no encontrado.")
            idx = [nombres_canales.index(picks)]

        elif isinstance(picks, list) and all(isinstance(p, str) for p in picks):  # Lista de nombres
            if not nombres_canales:
                raise ValueError("No se puede seleccionar por nombre sin info['canales'].")
            idx = []
            for ch in picks:
                if ch not in nombres_canales:
                    raise ValueError(f"Canal '{ch}' no encontrado.")
                idx.append(nombres_canales.index(ch))

        elif isinstance(picks, (list, np.ndarray)) and all(isinstance(p, int) for p in picks):  # Lista de índices
            if any(p < 0 or p >= n_canales for p in picks):
                raise ValueError("Índices fuera de rango.")
            idx = picks

        elif isinstance(picks, slice):
            idx = list(range(n_canales))[picks]

        else:
            raise ValueError("Formato de 'picks' inválido. Debe ser str, lista de nombres, lista de índices o slice.")

        # Seleccionar datos y nombres
        nueva_data = self.data[idx, :]

        nueva_info = self.info.copy() if self.info else {}
        if nombres_canales:
            nueva_info["canales"] = [nombres_canales[i] for i in idx]

        # Crear y retornar nueva instancia de RawSignal
        return RawSignal(
            data=nueva_data,
            sfreq=self.sfreq,
            info=nueva_info,
            anotaciones=self.anotaciones,
            first_samp=self.first_samp
        )

    def set_anotaciones(self, anotaciones):
        """
            Asocia un objeto de tipo 'Anotaciones' a la señal fisiológica.

        Parameters
        ----------
        anotaciones : Anotaciones
            Objeto de la clase 'Anotaciones' que contiene la información de los eventos.

        Raises
        ------
        TypeError
            Si el parámetro 'anotaciones' no es una instancia de la clase 'Anotaciones'.
        ValueError
            Si alguna anotación está fuera del rango temporal de la señal.
        """
        import pandas as pd

        # Validar tipo
        if not isinstance(anotaciones):#, Anotaciones): ## da error aca pq no tengo la clase Anotaciones definida
                                                    ## en este archivo, pero si la tuviera, validaria que el
                                                    ## parametro anotaciones sea una instancia de la clase 
                                                    # Anotaciones cuando esten juntos borrar el "):#"
            raise TypeError("El parámetro 'anotaciones' debe ser una instancia de la clase 'Anotaciones'.")

        df = anotaciones.get_annotations()
        if not isinstance(df, pd.DataFrame):
            raise ValueError("El objeto Anotaciones no contiene un DataFrame válido.")

        # Calcular duración total de la señal
        duracion_total = self.data.shape[1] / self.sfreq

        # Validar que todas las anotaciones estén dentro del rango
        for i, fila in df.iterrows():
            inicio = fila["Inicio (s)"]
            duracion = fila["Duración (s)"]
            if not (0 <= inicio <= duracion_total):
                raise ValueError(f"Anotación con inicio en {inicio}s fuera del rango (0 - {duracion_total:.2f}s).")
            if inicio + duracion > duracion_total:
                raise ValueError(f"Anotación desde {inicio}s con duración {duracion}s excede el final de la señal.")

        # Asignar si todo es válido
        self.anotaciones = anotaciones


    